# w04 — Baseline: Signal Checks, Rule, and Top-10 Review

Lane: **Structured Content Archetype Clustering**. This is the transparent, honestly-beatable rule my Week-5 clustering work needs to outperform.

## 1) Two signal checks (at least one flag-linked)

**Signal 1 — staleness behind FlyRank's refresh flags.** Does `days_since_last_update` actually relate to decline (`trend_direction == 'down'`)?

In [1]:
# Path assumes this notebook lives in work/notebooks/ — adjust if you moved it.
DATA_PATH = "../../data/raw/content_refresh_anonymized.csv"

import pandas as pd
df = pd.read_csv(DATA_PATH)

bins = [0, 90, 180, 365, 10000]
labels = ["<90d", "90-180d", "180-365d", "365d+"]
df["staleness_bucket"] = pd.cut(df["days_since_last_update"], bins=bins, labels=labels)

signal1_table = df.groupby("staleness_bucket", observed=True).agg(
    n=("content_id", "count"),
    decline_rate=("trend_direction", lambda s: (s == "down").mean())
).round(3)
print(signal1_table)

                      n  decline_rate
staleness_bucket                     
<90d              20655         0.512
90-180d            9171         0.611
180-365d            169         0.467
365d+                 5         0.600


**Verdict: MIXED.** Decline rate rises from `<90d` (0.512) to `90-180d` (0.611) as expected — but then *drops* at `180-365d` (0.467), lower than even the freshest bucket, and `365d+` has only 5 rows (too small to trust). Staleness alone doesn't cleanly predict decline in this data. This is exactly the kind of negative that saves a rule: it stops me from building a threshold purely on `days_since_last_update` and pushes me toward a signal that actually held up.

**Signal 2 — CTR vs. position behind the CTR-fix logic.** Does CTR really fall as position gets worse, the assumption a CTR-fix flag depends on?

In [2]:
has_pos = df[df["avg_position"] > 0].copy()  # avg_position==0 is a "no data" placeholder, not rank 0

pos_bins = [0, 3, 10, 20, 1000]
pos_labels = ["1-3", "4-10", "11-20", "21+"]
has_pos["position_bucket"] = pd.cut(has_pos["avg_position"], bins=pos_bins, labels=pos_labels)

signal2_table = has_pos.groupby("position_bucket", observed=True).agg(
    n=("content_id", "count"),
    mean_ctr=("ctr", "mean")
).round(3)
print(signal2_table)
print("\n(ctr is a x100 percentage per the data dictionary -- 2.71 means 2.71%, not 271%)")

                     n  mean_ctr
position_bucket                 
1-3               1141     2.714
4-10             11842     0.651
11-20             7273     0.323
21+               8539     0.211

(ctr is a x100 percentage per the data dictionary -- 2.71 means 2.71%, not 271%)


**Verdict: CONFIRMED.** CTR falls cleanly and monotonically as position worsens — 2.71% at positions 1–3, down to 0.65%, 0.32%, and 0.21% at 21+. Every bucket has thousands of rows, so this isn't a small-n fluke. This is the signal my rule leans on: a page whose actual CTR falls well below what its position tier normally earns is a real, verified anomaly worth a human look.

## 2) Encode one rule: score, one reason code, one action label

In [3]:
import numpy as np

# Expected CTR by tier, taken straight from the CONFIRMED signal check above
expected_ctr = has_pos.groupby("position_bucket", observed=True)["ctr"].mean().to_dict()
has_pos["expected_ctr"] = has_pos["position_bucket"].astype(str).map(expected_ctr).astype(float)
has_pos["ctr_gap"] = (has_pos["expected_ctr"] - has_pos["ctr"]).clip(lower=0)

# The rule in plain words: "A page is worth reviewing if it gets real traffic, holds a
# decent position, but earns noticeably less CTR than other pages at that same position."
visible = (has_pos["impressions_90d"] >= 500).astype(int)
good_position = (has_pos["avg_position"] <= 20).astype(int)
has_pos["score"] = visible * good_position * has_pos["ctr_gap"] * has_pos["impressions_90d"]

has_pos["reason_code"] = np.where(has_pos["score"] > 0, "low_ctr_for_position", "none")
has_pos["action"] = np.where(has_pos["score"] > 0, "rewrite_title_meta", "monitor")

print("Flagged rows:", (has_pos["score"] > 0).sum(), "/", len(has_pos))

Flagged rows: 9926 / 28795


In [4]:
import os
os.makedirs("../outputs", exist_ok=True)

queue = has_pos.sort_values("score", ascending=False)[
    ["content_id", "impressions_90d", "avg_position", "ctr", "expected_ctr",
     "score", "reason_code", "action"]
]
queue.to_csv("../outputs/baseline_action_score.csv", index=False)
print("Saved ../outputs/baseline_action_score.csv --", len(queue), "rows")

Saved ../outputs/baseline_action_score.csv -- 28795 rows


**Evaluate honestly with precision@K**, using `trend_direction == 'down'` as an evaluation-only proxy label (never a feature in the score above).

In [5]:
has_pos["declining_proxy"] = (has_pos["trend_direction"] == "down").astype(int)
base_rate = has_pos["declining_proxy"].mean()

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

print(f"base rate (share already declining): {base_rate:.3f}")
for k in [10, 20, 50]:
    p = precision_at_k(has_pos["score"].values, has_pos["declining_proxy"].values, k)
    print(f"precision@{k}: {p:.3f}")

base rate (share already declining): 0.564


precision@10: 0.300
precision@20: 0.200
precision@50: 0.320


**Honest read of these numbers:** precision@10/20/50 all land *below* the base rate of 0.564. That looks bad at first glance, but it's actually the correct, expected result once you think about what this rule is built to find versus what `declining_proxy` measures. The rule finds **CTR-underperformance relative to position** — which, looking at the top 10 below, tends to hit big, *stable or growing* pages sitting on a large but under-converted audience, not pages that are actively losing traffic. Those are two different problems. Reporting the honest number here — even though it's unflattering against this particular label — is the point: this baseline should be evaluated as a CTR-opportunity finder, not a decline-predictor, and precision@K makes that mismatch visible instead of hiding it.

## 3) Top-10 review

In [6]:
top10 = has_pos.sort_values("score", ascending=False).head(10)
print(top10[["content_id","impressions_90d","avg_position","ctr","expected_ctr",
             "trend_direction","action"]].to_string(index=False))

          content_id  impressions_90d  avg_position  ctr  expected_ctr trend_direction             action
content_8c19996aa890           509252           2.5 0.15      2.714303            down rewrite_title_meta
content_4c36c775b818           463103           2.3 0.41      2.714303            down rewrite_title_meta
content_8451fc6f034d           272144           2.3 0.03      2.714303              up rewrite_title_meta
content_44e481c8f55b           312694           1.4 0.65      2.714303          stable rewrite_title_meta
content_9532f197bbc8           309192           2.0 0.87      2.714303            down rewrite_title_meta
content_e12868d1f396           149712           2.9 0.07      2.714303          stable rewrite_title_meta
content_4a6607efcb46           128068           2.2 0.01      2.714303              up rewrite_title_meta
content_4fc39a2b8cf0           160959           2.6 0.69      2.714303          stable rewrite_title_meta
content_11900bd7941a           123561         

1. **content_8c19996aa890** — action: rewrite_title_meta. Why: 509k impressions at position 2.5 but only 0.15% CTR vs. an expected 2.71% — the single largest gap in the dataset. What would make it wrong: if this page's queries are heavily navigational/branded, low CTR could be normal (searchers already know the URL) rather than a real title/meta problem.
2. **content_4c36c775b818** — action: rewrite_title_meta. Why: 463k impressions, position 2.3, CTR 0.41% vs. expected 2.71%. What would make it wrong: a recent SERP feature (featured snippet, AI overview) could be siphoning clicks regardless of title quality — no title rewrite would fix that.
3. **content_8451fc6f034d** — action: rewrite_title_meta. Why: 272k impressions, position 2.3, near-zero CTR (0.03%). What would make it wrong: `trend_direction` is 'up' here — a page already gaining shouldn't be assumed broken; the low CTR might reflect an unusually competitive SERP, not a fixable on-page issue.
4. **content_44e481c8f55b** — action: rewrite_title_meta. Why: 312k impressions at the best position in this list (1.4), CTR 0.65% still well under the 2.71% expectation. What would make it wrong: position 1.4 pages often carry a rich snippet or knowledge panel that satisfies the query without a click — that's not a title problem.
5. **content_9532f197bbc8** — action: rewrite_title_meta. Why: 309k impressions, position 2.0, CTR 0.87%. What would make it wrong: if this is a hub/category page, lower CTR can be structural (users scan rather than click one result).
6. **content_e12868d1f396** — action: rewrite_title_meta. Why: 149k impressions, good position (2.9), CTR near zero (0.07%), `trend_direction` stable. What would make it wrong: stable trend with this large a CTR gap for this long suggests the gap may be permanent/structural, not a quick title fix — worth a manual look before assuming this is easily fixable.
7. **content_4a6607efcb46** — action: rewrite_title_meta. Why: 128k impressions, position 2.2, CTR effectively zero (0.01%). What would make it wrong: `trend_direction` is 'up' — an already-improving page with this profile might just have an unusual query mix (e.g. image or video-pack results) suppressing clicks independent of the title.
8. **content_4fc39a2b8cf0** — action: rewrite_title_meta. Why: 160k impressions, position 2.6, CTR 0.69%. What would make it wrong: same structural-SERP caveat as above — worth checking the actual SERP before committing review time.
9. **content_11900bd7941a** — action: rewrite_title_meta. Why: 123k impressions, position 2.8, CTR 0.41%. What would make it wrong: smaller absolute impressions than the pages above it — the ranking here is right, but the review payoff is proportionally smaller.
10. **content_03d2673b2553** — action: rewrite_title_meta. Why: 143k impressions, position 1.9 (near-best in the list), CTR 0.83%. What would make it wrong: this is the smallest CTR gap in the top 10 relative to its position tier — it's a weaker pick than the pages above it and could be bumped by a stronger candidate outside the top 10 if K were expanded.

## 4) Weak picks + leakage check

**Weak picks:** looking at `trend_direction` across the top 10 — most are 'stable' or 'up', not 'down'. That's the honest weak spot of this baseline: it's very good at finding big pages under-converting relative to their position, but it is *not* a decline-finder, and treating its top picks as "at risk" pages would be a mistake. Rows 6 and 9 in particular are the weakest of the ten — row 6 because a persistent stable gap this large may not be a quick-fix problem, and row 9 because its absolute traffic is meaningfully smaller than the rest of the list, so the same fix effort pays off less.

**Leakage check:** the score uses only `impressions_90d`, `avg_position`, and `ctr` — all observed-before-the-decision-point signals. `trend_direction` is used strictly for evaluation (the precision@K proxy label), never fed into the score itself. No product-decision flags (`health_score`, `priority_score`, etc.) are in this dataset at all, so there's nothing to accidentally leak from that direction.

## 5) Self-check

- [ ] Two signal checks with visible bucket tables and n, verdicts stated (MIXED for staleness, CONFIRMED for position→CTR)
- [ ] One rule encoded as a transparent score, with one reason code (`low_ctr_for_position`) and one action label (`rewrite_title_meta` / `monitor`)
- [ ] Ranked queue written to `work/outputs/baseline_action_score.csv` from this notebook
- [ ] precision@K computed and reported honestly, including the below-base-rate result and why that's the correct/expected outcome for what this rule targets
- [ ] Top 10 reviewed individually: action, why, what would make it wrong
- [ ] Weak picks named specifically (rows 6 and 9), not just a generic disclaimer
- [ ] No future-window or label-derived inputs in the score itself
- [ ] Notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere